In [6]:
from pybaseball import batting_stats_bref
import pandas as pd
import requests

In [80]:
# get LA sweet spot data stats from csv
launch_angle_data = pd.read_csv('launch_angle_data/savant_launch_angle_data.csv')

launch_angle_data = launch_angle_data[['last_name, first_name', 'sweet_spot_percent']]

In [ ]:
# get team wrc+ from fangraphs api

START = 2020
END = 2025

all_years = []

for year in range(START, END + 1):

    url = (
        f"https://www.fangraphs.com/api/leaders/major-league/data"
        f"?pos=all"
        f"&stats=bat"
        f"&lg=all"
        f"&qual=0"
        f"&season={year}"
        f"&season1={year}"
        f"&team=0,ts"
        f"&ind=0"
        f"&type=8"
        f"&pageitems=1000"
    )

    df = pd.read_json(url)

    rows = []

    for i in range(len(df)):

        team = df.iloc[i]["data"]   # one team dict

        rows.append({
            "Season": year,
            "Team": team["TeamNameAbb"],
            "wRC+": team["wRC+"]
        })

    all_years.append(pd.DataFrame(rows))

team_wrc = pd.concat(
    all_years,
    ignore_index=True
)

In [23]:
team_wrc

,Season,Team,wRC+
0,2020,LAD,120.221187
1,2020,SDP,116.439079
2,2020,CHW,111.135002
3,2020,ATL,119.817551
4,2020,NYM,122.744579
...,...,...,...
175,2025,WSN,92.809831
176,2025,LAA,91.598627
177,2025,PIT,82.200798
178,2025,CHW,88.475751


In [89]:
# figure out which players played for what team in each season

START = 2015
END = 2025

all_players = []

for year in range(START, END + 1):
    print(f"Loading {year}")

    # Baseball Reference player batting stats
    df = batting_stats_bref(year)
    
    print(df.iloc[2])

    df["Season"] = year
    
    # fix decoding errors
    df["Name"] = (
        df["Name"]
        .apply(lambda x: bytes(x, "utf-8").decode("unicode_escape"))
        .str.encode("latin1")
        .str.decode("utf-8")
    )

    all_players.append(
        df[[
            "Name",
            "Season",
            "Tm",
            "PA",
        ]]
    )

players = pd.concat(all_players, ignore_index=True)

print(players)

players.to_csv('player_team_data.csv')

player_team_year = pd.read_csv('player_team_data.csv', index_col=0)

Loading 2015
Name        Dustin Ackley
Age                    27
#days                3881
Lev                Maj-AL
Tm       New York,Seattle
G                     100
PA                    264
AB                    238
R                      28
H                      55
2B                     11
3B                      3
HR                     10
RBI                    30
BB                     18
IBB                     0
SO                     45
HBP                     1
SH                      3
SF                      4
GDP                     3
SB                      2
CS                      2
BA                  0.231
OBP                 0.284
SLG                 0.429
OPS                 0.712
mlbID              554429
Name: 3, dtype: object
Loading 2016
Name     Dustin Ackley
Age                 28
#days             3643
Lev             Maj-AL
Tm            New York
G                   25
PA                  70
AB                  61
R                    6
H               

In [83]:
player_team_year.head(3)
player_team_year[player_team_year['Name'] == "Dustin Ackley"]

,Name,Season,Tm,PA
2,Dustin Ackley,2015,"New York,Seattle",264
967,Dustin Ackley,2016,New York,70


In [77]:
team_wrc.head(3)

,Season,Team,wRC+
0,2020,LAD,120.221187
1,2020,SDP,116.439079
2,2020,CHW,111.135002


In [81]:
launch_angle_data.head(3)

,"last_name, first_name",sweet_spot_percent
0,"Colon, Bartolo",23.1
1,"Hunter, Torii",28.5
2,"Ortiz, David",34.8
